# Notebook 3 – Error Pattern Analysis & Concept Recommendation Mapping

This notebook analyses the relationship between error types, code patterns, and the pedagogical concept recommendations in CodeSense.

- Which Python constructs are most error-prone?
- What is the concept recommendation coverage for each error category?
- How do error messages cluster by language features?
- What are the most effective fix suggestions per category?

**Use:** Understanding this mapping helps improve the recommendation engine.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter
import re
import json

df = pd.read_csv('../dataset/error_dataset.csv')
print(f'Dataset: {len(df)} rows, {df["error_category"].nunique()} categories')
df.head(2)

In [ ]:
# ── Error type distribution per category ─────────────────────────────────────
print('Unique error_type values per category:')
for cat in sorted(df['error_category'].unique()):
    sub = df[df['error_category'] == cat]
    types = sub['error_type'].value_counts()
    print(f'\n{cat} ({len(sub)} samples):')
    for etype, cnt in types.head(5).items():
        pct = cnt / len(sub) * 100
        print(f'  {etype:35s}  {cnt:4d}  ({pct:.1f}%)')

In [ ]:
# ── Concept coverage: how many unique concepts does each category have? ───────
concept_coverage = df.groupby('error_category').agg(
    unique_concepts=('concept', 'nunique'),
    total_samples=('concept', 'count'),
    top_concept=('concept', lambda x: x.value_counts().index[0])
).reset_index()

print('Concept Coverage by Error Category:')
print(concept_coverage.to_string(index=False))

In [ ]:
# ── Bar chart: unique concepts per category ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
categories = concept_coverage['error_category']
n_concepts = concept_coverage['unique_concepts']
colors = plt.cm.Set2(np.linspace(0, 1, len(categories)))

bars = ax.barh(categories, n_concepts, color=colors, edgecolor='white')
for bar, n in zip(bars, n_concepts):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{n}', va='center', fontweight='bold')

ax.set_xlabel('Number of Unique Concepts')
ax.set_title('Concept Diversity per Error Category', fontweight='bold', fontsize=13)
ax.set_xlim(0, n_concepts.max() + 2)
plt.tight_layout()
plt.savefig('concept_coverage.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: concept_coverage.png')

In [ ]:
# ── Most common Python tokens/patterns in error-prone code ───────────────────
python_tokens = [
    'print', 'for', 'while', 'if', 'else', 'elif', 'def', 'class',
    'import', 'return', 'try', 'except', 'range', 'len', 'input',
    'list', 'dict', 'str', 'int', 'float', 'append', 'split', 'open'
]

token_freq = {}
for cat in sorted(df['error_category'].unique()):
    sub_code = ' '.join(df[df['error_category'] == cat]['code'].tolist())
    token_freq[cat] = {tok: sub_code.count(tok) for tok in python_tokens}

token_df = pd.DataFrame(token_freq).T
print('Python Token Frequency per Error Category (normalised by 250 samples):')
print((token_df / 250).round(2).to_string())

In [ ]:
# ── Heatmap: token frequency per category ────────────────────────────────────
top_tokens = token_df.sum().nlargest(12).index.tolist()
heatmap_data = (token_df[top_tokens] / 250).round(2)

fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(heatmap_data.values, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Avg occurrences per sample')

ax.set_xticks(range(len(top_tokens)))
ax.set_xticklabels(top_tokens, rotation=35, ha='right', fontsize=10)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index, fontsize=10)
ax.set_title('Python Token Frequency Heatmap (per error category)', fontweight='bold', fontsize=12)

for i in range(len(heatmap_data)):
    for j in range(len(top_tokens)):
        val = heatmap_data.values[i, j]
        color = 'white' if val > 1.5 else 'black'
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.savefig('token_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: token_heatmap.png')

In [ ]:
# ── Recommendation JSON audit ─────────────────────────────────────────────────
with open('../backend/recommendation/recommendations.json', 'r') as f:
    recommendations = json.load(f)

print('Recommendation entries in recommendations.json:')
print(f'  Total categories: {len(recommendations)}')
print()
for cat, rec in recommendations.items():
    n_topics = len(rec.get('topics', []))
    has_example = bool(rec.get('example'))
    has_practice = bool(rec.get('practice'))
    print(f'  {cat:22s} topics={n_topics}  example={has_example}  practice={has_practice}')

In [ ]:
# ── Category → concept → recommendation alignment check ───────────────────────
print('Category ↔ Recommendation Concept Alignment:')
for cat in sorted(df['error_category'].unique()):
    dataset_concepts = df[df['error_category'] == cat]['concept'].value_counts().head(3).index.tolist()
    rec_concept = recommendations.get(cat, {}).get('concept', 'NOT FOUND')
    rec_topics  = recommendations.get(cat, {}).get('topics', [])
    
    print(f'\n{cat}:')
    print(f'  Recommendation concept : {rec_concept}')
    print(f'  Dataset top concepts   : {dataset_concepts}')
    print(f'  Rec topics             : {rec_topics}')

In [ ]:
# ── Suggested_fix uniqueness analysis ─────────────────────────────────────────
print('Suggested Fix Diversity per Category (unique fix texts):')
for cat in sorted(df['error_category'].unique()):
    sub = df[df['error_category'] == cat]
    n_unique = sub['suggested_fix'].nunique()
    pct = n_unique / len(sub) * 100
    top_fix = sub['suggested_fix'].value_counts().index[0]
    print(f'  {cat:22s}  {n_unique:3d} unique ({pct:.0f}%)  |  Most common: "{top_fix[:60]}"')

In [ ]:
# ── Summary: recommendation readiness score ───────────────────────────────────
print('=== Recommendation Engine Readiness ===')
all_cats = sorted(df['error_category'].unique())
for cat in all_cats:
    rec = recommendations.get(cat, {})
    fields = ['concept', 'why', 'topics', 'example', 'practice', 'difficulty']
    filled = sum(1 for f in fields if rec.get(f))
    score = filled / len(fields) * 100
    status = '✅' if score == 100 else '⚠️'
    print(f'  {status} {cat:22s}  {filled}/{len(fields)} fields  ({score:.0f}% complete)')

missing = [c for c in all_cats if c not in recommendations]
if missing:
    print(f'\n⚠️  Categories without any recommendation: {missing}')
else:
    print('\n✅ All 8 error categories have full recommendations.')